In [0]:
%sql -- SETUP
CREATE SCHEMA IF NOT EXISTS workspace.sdp_bronze;
CREATE SCHEMA IF NOT EXISTS workspace.sdp_silver;
CREATE SCHEMA IF NOT EXISTS workspace.sdp_gold;

CREATE VOLUME IF NOT EXISTS
  workspace.sdp_bronze.source;

In [0]:
## ORDERS JSON
import json 

catalog = "workspace"  # Replace if necessary

source = f"/Volumes/{catalog}/sdp_bronze/source"
orders_path = f"{source}/orders"

orders = [
    {"order_id": 101, "customer_id": 1, "order_timestamp": 1760090400, "notifications": "Y"},
    {"order_id": 102, "customer_id": 2, "order_timestamp": 1760094000, "notifications": "N"},
    {"order_id": 103, "customer_id": 3, "order_timestamp": 1760097600, "notifications": "Y"},
    {"order_id": 104, "customer_id": 1, "order_timestamp": 1760101200, "notifications": "Y"},
    {"order_id": 105, "customer_id": 4, "order_timestamp": 1760176800, "notifications": "N"},
    {"order_id": 106, "customer_id": 2, "order_timestamp": 1760180400, "notifications": "Y"},
    {"order_id": 107, "customer_id": 5, "order_timestamp": 1760184000, "notifications": "N"},
    {"order_id": 108, "customer_id": 3, "order_timestamp": 1760263200, "notifications": "Y"}
]

import os
os.makedirs(orders_path, exist_ok=True)

with open(f"{orders_path}/00.json", "w") as f:
    for order in orders:
        f.write(json.dumps(order) + "\n")

print("Created 00.json with 8 orders")

Created 00.json with 8 orders


In [0]:
import json

new_orders = [
    {"order_id": 109, "customer_id": 6, "order_timestamp": 1760104800, "notifications": "Y"},
    {"order_id": 110, "customer_id": 7, "order_timestamp": 1760108400, "notifications": "N"},
    {"order_id": 111, "customer_id": 8, "order_timestamp": 1760187600, "notifications": "Y"},
    {"order_id": 112, "customer_id": 9, "order_timestamp": 1760266800, "notifications": "N"}
]

with open(f"{orders_path}/01.json", "w") as f:
    for order in new_orders:
        f.write(json.dumps(order) + "\n")

print("Added 01.json with 4 new orders")

Added 01.json with 4 new orders


In [0]:
## STATUS JSON
import json
import os

source_path = "/Volumes/workspace/sdp_bronze/source"
status_path = f"{source_path}/status"

os.makedirs(status_path, exist_ok=True)

# Use the original order timestamps.
orders = {
    101: 1760090400,
    102: 1760094000,
    103: 1760097600,
    104: 1760101200,
    105: 1760176800,
    106: 1760180400,
}

# Two status events for each order.
events = [
    (101, "placed", 0),
    (101, "delivered", 2),

    (102, "placed", 0),
    (102, "canceled", 1),

    (103, "placed", 0),
    (103, "delivered", 3),

    (104, "placed", 0),
    (104, "delivered", 1),

    (105, "placed", 0),
    (105, "canceled", 2),

    (106, "placed", 0),
    (106, "on the way", 1),
]

statuses = [
    {
        "order_id": order_id,
        "order_status": status,
        "status_timestamp": orders[order_id] + days * 86400
    }
    for order_id, status, days in events
]

file_path = f"{status_path}/00.json"

if os.path.exists(file_path):
    raise FileExistsError(
        "00.json already exists. Do not overwrite an ingested file."
    )

with open(file_path, "w") as f:
    for status in statuses:
        f.write(json.dumps(status) + "\n")

print(f"Created {len(statuses)} status records")


Created 12 status records


SOME VALIDATION FOR ORDERS

In [0]:
%sql
SELECT *
FROM workspace.sdp_gold.orders_by_date
ORDER BY order_date;

order_date,total_daily_orders
2025-10-10,4
2025-10-11,3
2025-10-12,1


In [0]:
%sql
SELECT *
FROM read_files(
    '/Volumes/workspace/sdp_bronze/source/status',
    format => 'json',
    inferColumnTypes => true
)
ORDER BY order_id, status_timestamp;

order_id,order_status,status_timestamp,_rescued_data
101,placed,1760090400,null
101,delivered,1760263200,null
102,placed,1760094000,null
102,canceled,1760180400,null
103,placed,1760097600,null
103,delivered,1760356800,null
104,placed,1760101200,null
104,delivered,1760187600,null
105,placed,1760176800,null
105,canceled,1760349600,null


In [0]:
%sql
SELECT *
FROM workspace.sdp_gold.orders_by_date
ORDER BY order_date;

order_date,total_daily_orders
2025-10-10,6
2025-10-11,4
2025-10-12,2


In [0]:
print(source)
/Volumes/workspace/sdp_bronze/source

/Volumes/workspace/sdp_bronze/source


In [0]:
%sql
SELECT
    timestamp,
    message
FROM event_log(
    TABLE(workspace.sdp_gold.orders_by_date)
)
WHERE event_type = 'planning_information'
ORDER BY timestamp DESC;

timestamp,message
2026-10-13T10:20:33.929Z,Flow 'workspace.sdp_gold.orders_by_date' has been planned to be executed as COMPLETE_RECOMPUTE.
2026-10-13T10:17:21.354Z,Flow 'workspace.sdp_gold.orders_by_date' has been planned to be executed as GROUP_AGGREGATE.
2026-10-13T10:16:55.091Z,Dry-run incrementalization check completed for flow 'workspace.sdp_gold.orders_by_date'.
2026-10-13T10:15:11.145Z,Flow 'workspace.sdp_gold.orders_by_date' has been planned to be executed as NO_OP.
2026-10-13T10:13:26.556Z,Flow 'workspace.sdp_gold.orders_by_date' has been planned to be executed as COMPLETE_RECOMPUTE.
2026-10-13T10:12:08.405Z,Dry-run incrementalization check completed for flow 'workspace.sdp_gold.orders_by_date'.
2026-10-13T10:08:13.440Z,Dry-run incrementalization check completed for flow 'workspace.sdp_gold.orders_by_date'.


In [0]:
## CUSTOMER JSON
import json
import os

folder = "/Volumes/workspace/sdp_bronze/source/customers_cdc"
os.makedirs(folder, exist_ok=True)

initial_customers = [
    (1, "Alice", "Helsinki"),
    (2, "Bob", "Espoo"),
    (3, "Charlie", "Tampere"),
    (4, "Diana", "Turku"),
    (5, "Emma", "Oulu"),
    (6, "Frank", "Lahti")
]

events = []

for i, (customer_id, name, city) in enumerate(initial_customers):
    events.append({
        "customer_id": customer_id,
        "name": name,
        "email": f"{name.lower()}@example.com",
        "city": city,
        "operation": "NEW",
        "timestamp": 1760000000 + i * 60
    })

path = f"{folder}/00.json"

if os.path.exists(path):
    raise FileExistsError("00.json already exists")

with open(path, "w") as f:
    for event in events:
        f.write(json.dumps(event) + "\n")

print("Created 6 initial customer events")


Created 6 initial customer events


In [0]:
%sql
SELECT *
FROM read_files(
    '/Volumes/workspace/sdp_bronze/source/customers_cdc',
    format => 'json',
    inferColumnTypes => true
)
ORDER BY timestamp;

city,customer_id,email,name,operation,timestamp,_rescued_data
Helsinki,1,alice@example.com,Alice,NEW,1760000000,null
Espoo,2,bob@example.com,Bob,NEW,1760000060,null
Tampere,3,charlie@example.com,Charlie,NEW,1760000120,null
Turku,4,diana@example.com,Diana,NEW,1760000180,null
Oulu,5,emma@example.com,Emma,NEW,1760000240,null
Lahti,6,frank@example.com,Frank,NEW,1760000300,null


In [0]:
%sql -- INTERMEDIATE VALIDATION
SELECT *
FROM workspace.sdp_silver.cdc_customers_silver
ORDER BY customer_id;

customer_id,name,email,city,event_time
1,Alice,alice@example.com,Helsinki,2025-10-09T08:53:20.000Z
2,Bob,bob@example.com,Espoo,2025-10-09T08:54:20.000Z
3,Charlie,charlie@example.com,Tampere,2025-10-09T08:55:20.000Z
4,Diana,diana@example.com,Turku,2025-10-09T08:56:20.000Z
5,Emma,emma@example.com,Oulu,2025-10-09T08:57:20.000Z
6,Frank,frank@example.com,Lahti,2025-10-09T08:58:20.000Z


In [0]:
## NEW CUSTOMERS CHANGES
import json
import os

changes = [
    {
        "customer_id": 1,
        "name": "Alice",
        "email": "alice@example.com",
        "city": "Espoo",
        "operation": "UPDATE",
        "timestamp": 1760001000
    },
    {
        "customer_id": 2,
        "name": "Bob",
        "email": "bob@example.com",
        "city": "Helsinki",
        "operation": "UPDATE",
        "timestamp": 1760001100
    },
    {
        "customer_id": 3,
        "name": "Charlie",
        "email": "charlie@example.com",
        "city": "Turku",
        "operation": "UPDATE",
        "timestamp": 1760001200
    },
    {
        "customer_id": 4,
        "name": "Diana",
        "email": "diana@example.com",
        "city": "Vantaa",
        "operation": "UPDATE",
        "timestamp": 1760001300
    },
    {
        "customer_id": 5,
        "name": None,
        "email": None,
        "city": None,
        "operation": "DELETE",
        "timestamp": 1760001400
    },
    {
        "customer_id": 6,
        "name": None,
        "email": None,
        "city": None,
        "operation": "DELETE",
        "timestamp": 1760001500
    }
]

path = f"{folder}/01.json"

if os.path.exists(path):
    raise FileExistsError("01.json already exists")

with open(path, "w") as f:
    for change in changes:
        f.write(json.dumps(change) + "\n")

print("Created 4 updates and 2 deletes")


Created 4 updates and 2 deletes


Final validation for customers

In [0]:
%sql
SELECT operation, COUNT(*) AS events
FROM workspace.sdp_bronze.cdc_customers_bronze_raw
GROUP BY operation
ORDER BY operation;

-- BRONZE

operation,events
DELETE,2
NEW,6
UPDATE,4


In [0]:
%sql
SELECT COUNT(*) AS active_customers
FROM workspace.sdp_silver.cdc_customers_silver;

-- SILVER

active_customers
4


In [0]:
%sql
SELECT *
FROM workspace.sdp_silver.cdc_customers_silver
WHERE customer_id IN (5, 6);

-- DELETED CUSTOMERS

customer_id,name,email,city,event_time


In [0]:
%sql
SELECT customer_id, operation, timestamp
FROM workspace.sdp_bronze.cdc_customers_bronze_raw
WHERE customer_id IN (1, 5)
ORDER BY customer_id, timestamp;

customer_id,operation,timestamp
1,NEW,1760000000
1,UPDATE,1760001000
5,NEW,1760000240
5,DELETE,1760001400
